# Demo: Create Agents to Research and Write an Article

This notebook demonstrates how to create a simple multi-agent multi-agent system using the crewAI framework.

## Preparing environment

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from crewai import Agent, Task, Crew, LLM

In [3]:
from IPython.display import Markdown

In [4]:
AGENT_ALLOW_DELEGATION = False
AGENT_VERBOSE = True
AGENT_TRACING = True

## Prepare locally hosted LLM

In [5]:
ollama_llm = LLM(
    model="ollama/mistral:latest",
    base_url="http://localhost:11434",
    temperature=0.7,
)

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_:

```
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:

```
varname = """line 1 of text
             line 2 of text
          """
```

is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [17]:
planner = Agent(
    role="Connect Planner",
    goal="Plan engaging and facturally accurate content on {topic}",
    backstory="You're working on planning a blog article"
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions."
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    llm=ollama_llm,
    allow_delegation=AGENT_ALLOW_DELEGATION,
    verbose=AGENT_VERBOSE
)

### Agent: Writer

In [18]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provided by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    llm=ollama_llm,
    allow_delegation=AGENT_ALLOW_DELEGATION,
    verbose=AGENT_VERBOSE
)

### Agent: Editor

In [19]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    llm=ollama_llm,
    allow_delegation=AGENT_ALLOW_DELEGATION,
    verbose=AGENT_VERBOSE
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [20]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
                    "with an outline, audience analysis, "
                    "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [21]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
        "3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
                    "in markdown format, ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [22]:
edit = Task(
    description=(
        "Proofread the given blog post for "
        "grammatical errors and alignment with the brand's voice."
    ),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: _For this simple example_, the tasks will be performed sequentially (i.e. they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=True` allows you to see execution logs.



In [29]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=AGENT_VERBOSE,
    tracing=AGENT_TRACING
)

NameError: name 'AGENT_TRACING' is not defined

## Running the Crew

In [26]:
from time import perf_counter

In [27]:
start = perf_counter()
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})
end = perf_counter()
print(f"\n\n{'='*100}\nTasks are finished in {end - start} seconds.\n{'='*100}\n\n")

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9c4e25b5-245b-4411-aefa-b754641a6f12                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Connect Planner                                                                                         │
│                                                                                                                 │
│  Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.            │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Connect Planner                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Content Plan for "Exploring the Latest Trends in Artificial Intelligence"                                      │
│                                                                                                                 │
│  Target Audience: Tech enthusiasts, business professionals, data scientists, and anyone interested in learning  │
│  about AI trends. Our primary focus will be on individuals who are seeking to make informed decisions about AI  │
│  technology and its applications.                                                                               │
│                                                                                                                 │
│  Content Outline:                                                                                               │
│  1. Introduction                                                                                                │
│     - Brief overview of artificial intelligence and its importance today.                                       │
│     - Explanation of the purpose of the article (exploring latest trends, key players, and news)                │
│                                                                                                                 │
│  2. Latest Trends in Artificial Intelligence                                                                    │
│     - Advancements in Machine Learning Algorithms (e.g., Generative Adversarial Networks, Transformers, etc.)   │
│     - Emerging AI Applications (e.g., autonomous vehicles, healthcare diagnostics, climate change modeling,     │
│  etc.)                                                                                                          │
│     - Discussion of the role of AI in various industries (e.g., finance, retail, healthcare, etc.)              │
│                                                                                                                 │
│  3. Key Players and Their Contributions to the Field of Artificial Intelligence                                 │
│     - Overview of major companies leading the way in AI research and development (e.g., Google, Microsoft,      │
│  IBM, etc.)                                                                                                     │
│     - Spotlight on notable AI researchers and their accomplishments                                             │
│                                                                                                                 │
│  4. Noteworthy News and Upcoming Developments                                                                   │
│     - Breaking news stories about recent advancements or partnerships in the field of AI                        │
│     - Discussion of upcoming conferences, events, or initiatives related to artificial intelligence             │
│                                                                                                                 │
│  5. Potential Challenges and Ethical Considerations                                                             │
│     - Overview of ethical concerns surrounding AI (e.g., bias, transparency, privacy, etc.)                     │
│     - Explanation of potential challenges in implementing AI technology                                         │
│                                                        

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 00425081-6735-4f2b-86cc-ec85c546515b                                                                     │
│  Agent: Connect Planner                                                                                         │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│  Task: 1. Use the content plan to craft a compelling blog post on Artificial Intelligence.                      │
│  2. Incorporate SEO keywords naturally.                                                                         │
│  3. Sections/Subtitles are properly named in an engaging manner.                                                │
│  4. Ensure the post is structured with an engaging introduction, insightful body, and a summarizing             │
│  conclusion.                                                                                                    │
│  5. Proofread for grammatical errors and alignment with the brand's voice.                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Task: Proofread the given blog post for grammatical errors and alignment with the brand's voice.               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```markdown                                                                                                    │
│  # Exploring the Latest Trends in Artificial Intelligence                                                       │
│  Welcome back to our blog, dear readers! Today, we're delving into the thrilling world of Artificial            │
│  Intelligence (AI), focusing on its latest trends, key players, and upcoming developments. If you're a tech     │
│  enthusiast, business professional, data scientist, or simply intrigued about AI, this post is crafted          │
│  specifically for you. Let's embark on this adventure together!                                                 │
│                                                                                                                 │
│  ## Latest Trends in Artificial Intelligence                                                                    │
│                                                                                                                 │
│  The AI landscape continues to evolve at an astonishing rate. Some significant advancements include the         │
│  refinement of Machine Learning Algorithms such as Generative Adversarial Networks (GANs) and Transformers,     │
│  which are transforming various industries. Other emerging applications include autonomous vehicles,            │
│  healthcare diagnostics, and climate change modeling.                                                           │
│                                                                                                                 │
│  ### Machine Learning Algorithms                                                                                │
│                                                                                                                 │
│  - **Generative Adversarial Networks (GANs)**: GANs are a class of algorithms that employ two neural networks   │
│  to generate content. They have been instrumental in creating convincing images, videos, and even music,        │
│  showcasing their potential for creative applications.                                                          │
│  - **Transformers**: Transformer models, such as BERT, have significantly enhanced the performance of natural   │
│  language processing tasks by understanding the context of words within a sentence. This breakthrough has led   │
│  to more sophisticated chatbots, language translation services, and sentiment analysis tools.                   │
│                                                                                                                 │
│  ### Emerging AI Applications                                                                                   │
│                                                                                                                 │
│  - **Autonomous Vehicles**: Self-driving cars are no longer just a futuristic concept. Companies like Tesla,    │
│  Waymo, and NVIDIA are heavily investing in this technology, with some cities already testing autonomous buses  │
│  and trucks.                                                                                                    │
│  - **Healthcare Diagnostics**: AI is being used to analyze medical images for early detection of diseases such  │
│  as cancer, paving the way for more accurate and timely

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: de157ee0-985d-4cde-a121-9b5ce8605d2f                                                                     │
│  Agent: Editor                                                                                                  │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



Tasks are finished in 28.814048008996906 seconds.




╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9c4e25b5-245b-4411-aefa-b754641a6f12                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ```markdown                                                                                      │
│  # Exploring the Latest Trends in Artificial Intelligence                                                       │
│  Welcome back to our blog, dear readers! Today, we're delving into the thrilling world of Artificial            │
│  Intelligence (AI), focusing on its latest trends, key players, and upcoming developments. If you're a tech     │
│  enthusiast, business professional, data scientist, or simply intrigued about AI, this post is crafted          │
│  specifically for you. Let's embark on this adventure together!                                                 │
│                                                                                                                 │
│  ## Latest Trends in Artificial Intelligence                                                                    │
│                                                                                                                 │
│  The AI landscape continues to evolve at an astonishing rate. Some significant advancements include the         │
│  refinement of Machine Learning Algorithms such as Generative Adversarial Networks (GANs) and Transformers,     │
│  which are transforming various industries. Other emerging applications include autonomous vehicles,            │
│  healthcare diagnostics, and climate change modeling.                                                           │
│                                                                                                                 │
│  ### Machine Learning Algorithms                                                                                │
│                                                                                                                 │
│  - **Generative Adversarial Networks (GANs)**: GANs are a class of algorithms that employ two neural networks   │
│  to generate content. They have been instrumental in creating convincing images, videos, and even music,        │
│  showcasing their potential for creative applications.                                                          │
│  - **Transformers**: Transformer models, such as BERT, have significantly enhanced the performance of natural   │
│  language processing tasks by understanding the context of words within a sentence. This breakthrough has led   │
│  to more sophisticated chatbots, language translation services, and sentiment analysis tools.                   │
│                                                                                                                 │
│  ### Emerging AI Applications                                                                                   │
│                                                                                                                 │
│  - **Autonomous Vehicles**: Self-driving cars are no longer just a futuristic concept. Companies like Tesla,    │
│  Waymo, and NVIDIA are heavily investing in this technology, with some cities already testing autonomous buses  │
│  and trucks.                                                                                                    │
│  - **Healthcare Diagnostics**: AI is being used to ana

- Display the results of your execution as markdown in the notebook.

In [28]:
from IPython.display import Markdown
Markdown(result.raw)

```markdown
# Exploring the Latest Trends in Artificial Intelligence
Welcome back to our blog, dear readers! Today, we're delving into the thrilling world of Artificial Intelligence (AI), focusing on its latest trends, key players, and upcoming developments. If you're a tech enthusiast, business professional, data scientist, or simply intrigued about AI, this post is crafted specifically for you. Let's embark on this adventure together!

## Latest Trends in Artificial Intelligence

The AI landscape continues to evolve at an astonishing rate. Some significant advancements include the refinement of Machine Learning Algorithms such as Generative Adversarial Networks (GANs) and Transformers, which are transforming various industries. Other emerging applications include autonomous vehicles, healthcare diagnostics, and climate change modeling.

### Machine Learning Algorithms

- **Generative Adversarial Networks (GANs)**: GANs are a class of algorithms that employ two neural networks to generate content. They have been instrumental in creating convincing images, videos, and even music, showcasing their potential for creative applications.
- **Transformers**: Transformer models, such as BERT, have significantly enhanced the performance of natural language processing tasks by understanding the context of words within a sentence. This breakthrough has led to more sophisticated chatbots, language translation services, and sentiment analysis tools.

### Emerging AI Applications

- **Autonomous Vehicles**: Self-driving cars are no longer just a futuristic concept. Companies like Tesla, Waymo, and NVIDIA are heavily investing in this technology, with some cities already testing autonomous buses and trucks.
- **Healthcare Diagnostics**: AI is being used to analyze medical images for early detection of diseases such as cancer, paving the way for more accurate and timely diagnosis.
- **Climate Change Modeling**: By analyzing vast amounts of data, AI can help predict climate patterns, monitor deforestation, and identify areas at risk from natural disasters, enabling us to take proactive measures against environmental degradation.

## Key Players and Their Contributions to the Field of Artificial Intelligence

Several major companies are leading the way in AI research and development. Google, Microsoft, IBM, and many others are investing billions of dollars in AI technology. Notable AI researchers, such as Geoffrey Hinton and Yoshua Bengio, have made groundbreaking contributions to the field. Their work has laid the foundation for machine learning algorithms and deep neural networks that underpin AI's success today.

## Noteworthy News and Upcoming Developments

Recent advancements in AI include Google's breakthrough in quantum computing and Microsoft's partnership with the European Union to develop AI for a more sustainable future. Upcoming conferences, such as NeurIPS (Conference on Neural Information Processing Systems) and TechCrunch Disrupt, will undoubtedly bring forth exciting announcements about the latest trends in AI.

## Potential Challenges and Ethical Considerations

While AI boasts immense potential benefits, it also raises ethical concerns. Issues such as bias, transparency, privacy, and accountability must be addressed to ensure that AI is developed and deployed responsibly. It's crucial for businesses, governments, and researchers to prioritize these challenges and work together to create a future where AI serves humanity rather than harms it.

## Call to Action

Stay curious, readers! The world of Artificial Intelligence is vast and ever-evolving. To learn more about AI and its applications, we recommend exploring websites like Coursera, edX, Kaggle, or TensorFlow. These resources offer a wealth of information, courses, and projects to help you dive deeper into the fascinating realm of artificial intelligence.
```

This post is structured with an engaging introduction, insightful body, and a summarizing conclusion. It naturally incorporates SEO keywords and is aligned with the brand's voice. The content has been proofread for grammatical errors.